# DriveSight — Évaluation finale et comparaison YOLO

Évalue les poids déjà entraînés sur le split `test`. Cette étape ne réentraîne aucun modèle.

In [2]:
%pip install pandas
from pathlib import Path
import os, csv
import torch, yaml
import pandas as pd
from ultralytics import YOLO


  Using cached pandas-3.1.0-cp311-cp311-win_amd64.whl.metadata (18 kB)
Using cached pandas-3.1.0-cp311-cp311-win_amd64.whl (11.0 MB)
Note: you may need to restart the kernel to use updated packages.


In [4]:
from pathlib import Path
import os
import csv
import torch
import yaml
import pandas as pd
from ultralytics import YOLO

REPO = Path(r"C:\Users\Kenza\DriveSight")
DATA_YAML = REPO / "data" / "yolo" / "data.yaml"
RUNS_DIR = REPO / "runs"

if not REPO.is_dir():
    raise FileNotFoundError(f"Dépôt introuvable : {REPO}")

if not DATA_YAML.is_file():
    raise FileNotFoundError(f"Dataset introuvable : {DATA_YAML}")

os.chdir(REPO)

print("Dépôt :", REPO)
print("Dataset trouvé :", DATA_YAML.is_file())
print("YOLOv8s :", (RUNS_DIR / "bdd_yolov8s" / "weights" / "best.pt").is_file())
print("YOLO11s :", (RUNS_DIR / "bdd_yolo11s" / "weights" / "best.pt").is_file())
print("GPU disponible :", torch.cuda.is_available())

Dépôt : C:\Users\Kenza\DriveSight
Dataset trouvé : True
YOLOv8s : True
YOLO11s : True
GPU disponible : True


## 1. Vérifier les poids disponibles

In [5]:
MODEL_PATHS = {
    "YOLO11s": RUNS_DIR / "bdd_yolo11s" / "weights" / "best.pt",
    "YOLOv8s": RUNS_DIR / "bdd_yolov8s" / "weights" / "best.pt",
}
for name, path in MODEL_PATHS.items():
    print(f"{name}: {path} | disponible : {path.is_file()}")


YOLO11s: C:\Users\Kenza\DriveSight\runs\bdd_yolo11s\weights\best.pt | disponible : True
YOLOv8s: C:\Users\Kenza\DriveSight\runs\bdd_yolov8s\weights\best.pt | disponible : True


## 2. Évaluation sur le test set

In [6]:
rows = []
for model_name, weights_path in MODEL_PATHS.items():
    if not weights_path.is_file():
        print(f"IGNORÉ : {model_name}, poids absents : {weights_path}")
        continue

    print(f"\\n===== Test final : {model_name} =====")
    model = YOLO(str(weights_path))
    metrics = model.val(
        data=str(DATA_YAML),
        split="test",
        imgsz=640,
        batch=4,
        device=0,
        plots=True,
        project=str(RUNS_DIR),
        name=f"test_{model_name.lower()}",
        exist_ok=True
    )
    rows.append({
        "model": model_name,
        "weights": str(weights_path.relative_to(REPO)),
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "mAP50": float(metrics.box.map50),
        "mAP50_95": float(metrics.box.map),
    })
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if not rows:
    raise FileNotFoundError(
        "Aucun poids trouvé. Exécutez train_yolo11s.ipynb et/ou train_yolov8s.ipynb d'abord."
    )

comparison = pd.DataFrame(rows)
display(comparison.style.format({
    "precision":"{:.4f}", "recall":"{:.4f}", "mAP50":"{:.4f}", "mAP50_95":"{:.4f}"
}))


\n===== Test final : YOLO11s =====
Ultralytics 8.4.190  Python-3.11.9 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
YOLO11s summary (fused): 100 layers, 9,416,283 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access  (ping: 0.1±0.0 ms, read: 100.9±22.0 MB/s, size: 60.5 KB)
val: Scanning C:\Users\Kenza\DriveSight\data\yolo\labels\test... 375 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 375/375 948.5it/s 0.4s0.2s
val: New cache created: C:\Users\Kenza\DriveSight\data\yolo\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 94/94 4.4it/s 21.2s<0.0s9
                   all        375       7351      0.545      0.446      0.465      0.258
            pedestrian        139        594        0.6      0.515      0.554      0.267
                 rider         21         31       0.39      0.165      0.295      0.151
                   car        370       3992      0.674      0

,model,weights,precision,recall,mAP50,mAP50_95
0,YOLO11s,runs\bdd_yolo11s\weights\best.pt,0.5447,0.4460,0.4652,0.2577
1,YOLOv8s,runs\bdd_yolov8s\weights\best.pt,0.5672,0.4415,0.4583,0.2493


## 3. Enregistrer les résultats

In [7]:
RESULTS_DIR = REPO / "experiments" / "yolo_model_comparison" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
csv_path = RESULTS_DIR / "comparison.csv"
comparison.to_csv(csv_path, index=False)
print("Comparaison sauvegardée :", csv_path)


Comparaison sauvegardée : C:\Users\Kenza\DriveSight\experiments\yolo_model_comparison\results\comparison.csv


## Interprétation

- La **précision** mesure la part des détections prédites qui sont correctes.
- Le **rappel** mesure la part des objets annotés qui sont retrouvés.
- **mAP50** utilise un seuil IoU de 0,50.
- **mAP50-95** moyenne l'AP sur plusieurs seuils IoU et est plus exigeante.

Comparer uniquement les résultats obtenus avec le même dataset, le même split, et des paramètres d'évaluation identiques.